#Preparación del Dashboard

###Definicion de KPIs

In [0]:
%sql
-- Utilizamos las consultas sql
CREATE OR REPLACE TABLE kpis_medicos AS

-- Se calcula las sumas básicas agrupadas 
WITH Metricas_Base AS (
    SELECT 
        nom_sucursal,
        especialidad_medica,
        COUNT(*) AS total_citas,
        SUM(CASE WHEN estado_cita = 'Completada' THEN pago_total ELSE 0 END) AS ingresos,
        SUM(CASE WHEN estado_cita = 'Cancelada' THEN 1 ELSE 0 END) AS canceladas,
        SUM(CASE WHEN estado_cita = 'Completada' THEN 1 ELSE 0 END) AS completadas
    FROM citas_pmm_limpioV3
    GROUP BY nom_sucursal, especialidad_medica
)

-- Se calculan los porcentajes y promedios finales
SELECT 
    nom_sucursal,
    especialidad_medica,
    total_citas AS citas,
    ingresos,
    -- Calculamos la tasa de cancelación (%)
    ROUND((canceladas / total_citas) * 100, 2) AS tasa_cancelacion,
    -- Calculamos el ticket promedio (evitando divisiones por cero con NULLIF)
    ROUND(ingresos / NULLIF(completadas, 0), 2) AS ticket_promedio
FROM Metricas_Base;


num_affected_rows,num_inserted_rows


###Mostramos los resultados para verificar

In [0]:
# Verificamos que la tabla se guardó correctamente y vemos los primeros registros
df_kpis = spark.table("kpis_medicos")
display(df_kpis)

nom_sucursal,especialidad_medica,citas,ingresos,tasa_cancelacion,ticket_promedio
PMM San Francisco,Odontología,277,12167.800000000005,19.49,54.56
PMM El Dorado,Cardiología,85,6572.52,20.0,96.65
PMM Costa del Este,Otorrinolaringología,225,16340.240000000002,15.11,85.55
PMM Costa del Este,Oftalmología,227,14535.849999999988,18.94,79.0
PMM El Dorado,Geriatría,28,1692.3200000000002,17.86,73.58
PMM San Francisco,Neurología,176,17801.94,18.18,123.62
PMM Tocumen,Radiología,79,3898.8300000000004,24.05,64.98
PMM Costa del Este,Ginecología,70,4583.350000000001,17.14,79.02
PMM Brisas del Golf,Pediatría,29,1240.0600000000002,17.24,51.67
PMM El Dorado,Urología,74,3625.94,18.92,60.43
